# Notebook #7 — EMA Pullback Reaction Strategy
## استراتژی واکنش به EMA در Pullback — XAUUSD (H1 + M5)

---

### فلسفه EMA Pullback

در یک روند قوی، قیمت به طور مرتب به EMA برمی‌گردد (Pullback) و مجدداً در جهت روند حرکت می‌کند.

**ما فقط واکنش قیمت به EMA را معامله می‌کنیم، نه پیش‌بینی ادامه روند.**

### شرایط ورود:

**BUY (Pullback در روند صعودی):**
1. روند H1 صعودی (قیمت بالای EMA21)
2. روند H1 قوی (EMA21 > EMA50)
3. M5: Pullback به EMA21
4. M5: کندل واکنش صعودی (Reaction Candle)
5. M5: EMA21 خود در جهت صعودی است

**SELL (Pullback در روند نزولی):**
1. روند H1 نزولی (قیمت زیر EMA21)
2. روند H1 قوی (EMA21 < EMA50)
3. M5: Pullback به EMA21
4. M5: کندل واکنش نزولی
5. M5: EMA21 خود در جهت نزولی است

### چند EMA را آزمایش می‌کنیم:
- EMA 9 (سریع)
- EMA 21 (متوسط)
- EMA 50 (کُند)
- ترکیب: EMA 21 (entry) + EMA 50 (trend)

---

| پارامتر | مقدار |
|---|---|
| نماد | XAUUSD |
| TF روند | H1 |
| TF ورود | M5 |
| EMA‌ها | 9, 21, 50 |
| Risk/Reward | 1:2 |

## Step 1 — Imports & Configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Dict, Tuple

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

SYMBOL        = 'XAUUSD'
DATA_DIR      = Path('./data')
LOOKBACK_DAYS = 30

# ── EMA Configuration ────────────────────────────────────────────────────────
EMA_FAST      = 9
EMA_MID       = 21
EMA_SLOW      = 50

# ── Entry EMA (M5 pullback target) ───────────────────────────────────────────
ENTRY_EMA     = EMA_MID    # pullback to this EMA on M5
TREND_EMA     = EMA_SLOW   # this determines H1 trend direction

# ── Pullback Detection ───────────────────────────────────────────────────────
PULLBACK_TOL  = 0.5        # price within this distance of EMA = touching
MOMENTUM_MIN  = 3          # minimum M5 bars in trend direction before pullback
PULLBACK_MAX_BARS = 20     # maximum bars for pullback to reach EMA

# ── Reaction Confirmation ────────────────────────────────────────────────────
REACTION_CANDLE_MIN_BODY = 0.3  # body >= 30% of range
SWING_WINDOW  = 5
SWING_LB_H    = 6

# ── Risk ─────────────────────────────────────────────────────────────────────
RISK_REWARD   = 2.0
SL_BUFFER     = 0.3
MAX_TRADE_BARS= 144

print('EMA Pullback Reaction — Config loaded.')
print(f'  Trend EMA  : EMA{TREND_EMA} on H1')
print(f'  Entry EMA  : EMA{ENTRY_EMA} on M5')
print(f'  Fast EMA   : EMA{EMA_FAST} on M5 (momentum)')
print(f'  Risk/Reward: 1:{RISK_REWARD}')

EMA Pullback Reaction — Config loaded.
  Trend EMA  : EMA50 on H1
  Entry EMA  : EMA21 on M5
  Fast EMA   : EMA9 on M5 (momentum)
  Risk/Reward: 1:2.0


## Step 2 — Load Data

In [2]:
def load_ohlcv(symbol: str, tf: str, lookback_days: int) -> pd.DataFrame:
    path = DATA_DIR / symbol / tf / 'ohlcv.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.sort_values('time').reset_index(drop=True)
    keep = ['time', 'open', 'high', 'low', 'close', 'tick_volume']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=lookback_days)
    return df[df['time'] >= cutoff].copy().reset_index(drop=True)


df_h1 = load_ohlcv(SYMBOL, 'H1', LOOKBACK_DAYS)
df_m5 = load_ohlcv(SYMBOL, 'M5', LOOKBACK_DAYS)

print(f'H1: {len(df_h1):,} bars')
print(f'M5: {len(df_m5):,} bars')

H1: 490 bars
M5: 5,887 bars


## Step 3 — EMA Calculations

همه EMAها بر روی هر دو تایم‌فریم محاسبه می‌شوند:

In [3]:
def add_emas(df: pd.DataFrame) -> pd.DataFrame:
    """Add EMA9, EMA21, EMA50 to dataframe."""
    df = df.copy()
    df[f'ema{EMA_FAST}']  = df['close'].ewm(span=EMA_FAST,  adjust=False).mean()
    df[f'ema{EMA_MID}']   = df['close'].ewm(span=EMA_MID,   adjust=False).mean()
    df[f'ema{EMA_SLOW}']  = df['close'].ewm(span=EMA_SLOW,  adjust=False).mean()

    # Trend on H1: above/below TREND_EMA
    df['trend_up']  = df['close'] > df[f'ema{TREND_EMA}']
    df['trend_dn']  = df['close'] < df[f'ema{TREND_EMA}']

    # EMA slope (direction)
    df[f'ema{EMA_MID}_slope']  = df[f'ema{EMA_MID}'].diff()
    df[f'ema{EMA_SLOW}_slope'] = df[f'ema{EMA_SLOW}'].diff()

    # EMA alignment (fast > mid > slow = strong bull)
    df['ema_aligned_bull'] = (
        df[f'ema{EMA_FAST}'] > df[f'ema{EMA_MID}'] and
        df[f'ema{EMA_MID}']  > df[f'ema{EMA_SLOW}']
    ).astype(bool) if False else (
        (df[f'ema{EMA_FAST}'] > df[f'ema{EMA_MID}']) &
        (df[f'ema{EMA_MID}']  > df[f'ema{EMA_SLOW}'])
    )
    df['ema_aligned_bear'] = (
        (df[f'ema{EMA_FAST}'] < df[f'ema{EMA_MID}']) &
        (df[f'ema{EMA_MID}']  < df[f'ema{EMA_SLOW}'])
    )

    return df


df_h1 = add_emas(df_h1)
df_m5 = add_emas(df_m5)

print('EMAs added:')
for tf_name, df in [('H1', df_h1), ('M5', df_m5)]:
    current = df.iloc[-1]
    print(f'  {tf_name}: Close={current["close"]:.2f}  '
          f'EMA{EMA_FAST}={current[f"ema{EMA_FAST}"]:.2f}  '
          f'EMA{EMA_MID}={current[f"ema{EMA_MID}"]:.2f}  '
          f'EMA{EMA_SLOW}={current[f"ema{EMA_SLOW}"]:.2f}')

print(f'\nH1 Trend (EMA{TREND_EMA}):')
h1_trend_up_pct = df_h1['trend_up'].mean() * 100
print(f'  Bullish: {h1_trend_up_pct:.0f}%  Bearish: {100-h1_trend_up_pct:.0f}%')

EMAs added:
  H1: Close=4540.01  EMA9=4551.34  EMA21=4573.38  EMA50=4616.58
  M5: Close=4540.01  EMA9=4540.00  EMA21=4543.45  EMA50=4547.46

H1 Trend (EMA50):
  Bullish: 38%  Bearish: 62%


## Step 4 — H1 Trend Function

In [4]:
def get_h1_trend(df_h1: pd.DataFrame, query_time: pd.Timestamp) -> int:
    """
    Get H1 trend at or before query_time.
    Returns: 1=UP, -1=DOWN, 0=NEUTRAL
    Anti-lookahead: uses only bars with time <= query_time.
    """
    past = df_h1[df_h1['time'] <= query_time]
    if past.empty:
        return 0
    last = past.iloc[-1]

    ema_fast  = last[f'ema{EMA_FAST}']
    ema_mid   = last[f'ema{EMA_MID}']
    ema_slow  = last[f'ema{EMA_SLOW}']
    close     = last['close']
    slope_mid = last[f'ema{EMA_MID}_slope']

    # Strong uptrend: close > mid > slow EMA, EMA slope positive
    if (close > ema_mid > ema_slow and slope_mid > 0):
        return 1
    # Strong downtrend
    elif (close < ema_mid < ema_slow and slope_mid < 0):
        return -1
    # Weak uptrend (at least above TREND_EMA)
    elif close > ema_slow:
        return 1
    elif close < ema_slow:
        return -1
    else:
        return 0


def get_swing_sl(df_m5: pd.DataFrame, entry_time: pd.Timestamp,
                direction: str, window: int = SWING_WINDOW,
                lb_h: float = SWING_LB_H) -> float:
    start = entry_time - pd.Timedelta(hours=lb_h)
    lb    = df_m5[(df_m5['time'] >= start) & (df_m5['time'] < entry_time)]
    if lb.empty:
        return np.nan
    arr = lb['low'].values if direction == 'BUY' else lb['high'].values
    n   = len(arr)
    if n < window * 2 + 1:
        return float(arr.min()) if direction == 'BUY' else float(arr.max())
    out = np.full(n, np.nan)
    for i in range(window, n - window):
        nb = np.concatenate([arr[i-window:i], arr[i+1:i+window+1]])
        if direction == 'BUY' and arr[i] <= nb.min():
            out[i] = arr[i]
        elif direction == 'SELL' and arr[i] >= nb.max():
            out[i] = arr[i]
    valid = [v for v in out if not np.isnan(v)]
    if not valid:
        return float(arr.min()) if direction == 'BUY' else float(arr.max())
    return valid[-1]


print('Trend and swing functions defined.')

Trend and swing functions defined.


## Step 5 — Pullback Detection State Machine

```
[IDLE]
  ├─ H1 trend UP + N bull M5 bars → MOMENTUM_UP
  └─ H1 trend DN + N bear M5 bars → MOMENTUM_DN

[MOMENTUM_UP]
  ├─ M5 price touches EMA21 from above → PULLBACK_AT_EMA_UP
  └─ trend reverses → IDLE

[PULLBACK_AT_EMA_UP]
  ├─ Bullish reaction candle + EMA21 slope up → ENTRY BUY
  ├─ Close below EMA21 - tolerance → cancel → IDLE
  └─ timeout → IDLE

(Mirror for SELL)
```

In [5]:
def simulate_trade(
    df: pd.DataFrame,
    entry_idx: int,
    direction: str,
    entry: float,
    sl: float,
    tp: float,
) -> dict:
    bars = df.iloc[entry_idx + 1 : entry_idx + MAX_TRADE_BARS + 1]
    for i, bar in enumerate(bars.itertuples(), 1):
        if direction == 'BUY':
            if bar.low <= sl:
                return {'result': 'SL', 'exit_price': sl, 'exit_time': bar.time, 'pnl_r': -1.0, 'bars_held': i}
            if bar.high >= tp:
                return {'result': 'TP', 'exit_price': tp, 'exit_time': bar.time, 'pnl_r': RISK_REWARD, 'bars_held': i}
        else:
            if bar.high >= sl:
                return {'result': 'SL', 'exit_price': sl, 'exit_time': bar.time, 'pnl_r': -1.0, 'bars_held': i}
            if bar.low <= tp:
                return {'result': 'TP', 'exit_price': tp, 'exit_time': bar.time, 'pnl_r': RISK_REWARD, 'bars_held': i}
    if not bars.empty:
        last = bars.iloc[-1]
        risk = abs(entry - sl)
        pnl  = ((last['close'] - entry) / risk if direction == 'BUY'
                else (entry - last['close']) / risk)
        return {'result': 'OPEN', 'exit_price': round(last['close'], 2),
                'exit_time': last['time'], 'pnl_r': round(pnl, 3), 'bars_held': len(bars)}
    return {'result': 'OPEN', 'exit_price': entry,
            'exit_time': df.iloc[entry_idx]['time'], 'pnl_r': 0.0, 'bars_held': 0}


def is_reaction_bullish(bar: pd.Series) -> bool:
    """Bullish reaction at EMA: close bullish, meaningful body."""
    rng  = bar['high'] - bar['low']
    body = abs(bar['close'] - bar['open'])
    return (bar['close'] > bar['open'] and
            rng > 0 and
            body / rng >= REACTION_CANDLE_MIN_BODY)


def is_reaction_bearish(bar: pd.Series) -> bool:
    rng  = bar['high'] - bar['low']
    body = abs(bar['close'] - bar['open'])
    return (bar['close'] < bar['open'] and
            rng > 0 and
            body / rng >= REACTION_CANDLE_MIN_BODY)


def run_ema_pullback_backtest(
    df_m5: pd.DataFrame,
    df_h1: pd.DataFrame,
    entry_ema: int = ENTRY_EMA,
) -> pd.DataFrame:
    """
    State machine for EMA Pullback Reaction.
    One trade per 'swing' (no two trades within 10 bars).
    """
    trades   = []
    used_idx = set()

    ema_col   = f'ema{entry_ema}'
    slope_col = f'ema{entry_ema}_slope'

    IDLE           = 0
    MOMENTUM_UP    = 1
    MOMENTUM_DN    = 2
    PULLBACK_UP    = 3
    PULLBACK_DN    = 4

    state          = IDLE
    momentum_count = 0
    pullback_count = 0

    for i in range(TREND_EMA + 5, len(df_m5) - 1):
        bar  = df_m5.iloc[i]
        ema  = bar[ema_col]
        slope= bar[slope_col]

        if any(abs(i - u) < 5 for u in used_idx):
            continue

        if np.isnan(ema):
            continue

        # Get H1 trend
        h1_trend = get_h1_trend(df_h1, bar['time'])

        close = bar['close']
        prev  = df_m5.iloc[i - 1]

        if state == IDLE:
            if h1_trend == 1 and close > ema and prev['close'] > prev[ema_col]:
                momentum_count = 1
                state          = MOMENTUM_UP
            elif h1_trend == -1 and close < ema and prev['close'] < prev[ema_col]:
                momentum_count = 1
                state          = MOMENTUM_DN

        elif state == MOMENTUM_UP:
            if h1_trend != 1:
                state = IDLE; momentum_count = 0; continue
            if close > ema:
                momentum_count += 1
            else:
                # Price came back to EMA — check if touching from above
                if (bar['low'] <= ema + PULLBACK_TOL and
                        close >= ema - PULLBACK_TOL):
                    state          = PULLBACK_UP
                    pullback_count = 0
                elif close < ema - PULLBACK_TOL * 2:
                    state = IDLE; momentum_count = 0  # broke below EMA

        elif state == MOMENTUM_DN:
            if h1_trend != -1:
                state = IDLE; momentum_count = 0; continue
            if close < ema:
                momentum_count += 1
            else:
                if (bar['high'] >= ema - PULLBACK_TOL and
                        close <= ema + PULLBACK_TOL):
                    state          = PULLBACK_DN
                    pullback_count = 0
                elif close > ema + PULLBACK_TOL * 2:
                    state = IDLE; momentum_count = 0

        elif state == PULLBACK_UP:
            pullback_count += 1

            if pullback_count > PULLBACK_MAX_BARS:
                state = IDLE; momentum_count = 0; continue

            # Still near EMA from above
            near_ema = bar['low'] <= ema + PULLBACK_TOL

            if not near_ema and close < ema - PULLBACK_TOL:
                state = IDLE; momentum_count = 0; continue  # gave up

            # Reaction candle: bullish close, EMA slope positive
            if (is_reaction_bullish(bar) and
                    close > ema and
                    slope > 0):
                # ENTRY BUY
                entry = close
                sl_swing = get_swing_sl(df_m5, bar['time'], 'BUY')
                sl   = min(sl_swing, ema - PULLBACK_TOL) - SL_BUFFER
                risk = entry - sl
                if risk <= 0:
                    continue
                tp = entry + RISK_REWARD * risk
                outcome = simulate_trade(df_m5, i, 'BUY', entry, sl, tp)
                trades.append({
                    'direction'    : 'BUY',
                    'entry_time'   : bar['time'],
                    'entry_price'  : round(entry, 2),
                    'sl'           : round(sl, 2),
                    'tp'           : round(tp, 2),
                    'risk'         : round(risk, 2),
                    'ema_at_entry' : round(ema, 2),
                    'ema_period'   : entry_ema,
                    'h1_trend'     : h1_trend,
                    'momentum_bars': momentum_count,
                    'pullback_bars': pullback_count,
                    **outcome,
                })
                used_idx.add(i)
                state = IDLE; momentum_count = 0; pullback_count = 0

        elif state == PULLBACK_DN:
            pullback_count += 1

            if pullback_count > PULLBACK_MAX_BARS:
                state = IDLE; momentum_count = 0; continue

            near_ema = bar['high'] >= ema - PULLBACK_TOL

            if not near_ema and close > ema + PULLBACK_TOL:
                state = IDLE; momentum_count = 0; continue

            if (is_reaction_bearish(bar) and
                    close < ema and
                    slope < 0):
                entry    = close
                sl_swing = get_swing_sl(df_m5, bar['time'], 'SELL')
                sl   = max(sl_swing, ema + PULLBACK_TOL) + SL_BUFFER
                risk = sl - entry
                if risk <= 0:
                    continue
                tp = entry - RISK_REWARD * risk
                outcome = simulate_trade(df_m5, i, 'SELL', entry, sl, tp)
                trades.append({
                    'direction'    : 'SELL',
                    'entry_time'   : bar['time'],
                    'entry_price'  : round(entry, 2),
                    'sl'           : round(sl, 2),
                    'tp'           : round(tp, 2),
                    'risk'         : round(risk, 2),
                    'ema_at_entry' : round(ema, 2),
                    'ema_period'   : entry_ema,
                    'h1_trend'     : h1_trend,
                    'momentum_bars': momentum_count,
                    'pullback_bars': pullback_count,
                    **outcome,
                })
                used_idx.add(i)
                state = IDLE; momentum_count = 0; pullback_count = 0

    return pd.DataFrame(trades) if trades else pd.DataFrame()


print('EMA Pullback backtest engine ready.')

EMA Pullback backtest engine ready.


In [6]:
trades_df = run_ema_pullback_backtest(df_m5, df_h1, entry_ema=ENTRY_EMA)

if trades_df.empty:
    print('No trades generated.')
else:
    print(f'Total trades: {len(trades_df)}')
    print(f'  BUY  : {(trades_df["direction"]=="BUY").sum()}')
    print(f'  SELL : {(trades_df["direction"]=="SELL").sum()}')
    print(f'  TP   : {(trades_df["result"]=="TP").sum()}')
    print(f'  SL   : {(trades_df["result"]=="SL").sum()}')
    display(trades_df.head(5))

Total trades: 65
  BUY  : 28
  SELL : 37
  TP   : 24
  SL   : 38


,direction,entry_time,entry_price,sl,tp,risk,ema_at_entry,ema_period,h1_trend,momentum_bars,pullback_bars,result,exit_price,exit_time,pnl_r,bars_held
0,BUY,2026-04-16 22:15:00+00:00,4793.1000,4786.3900,4806.5200,6.7100,4790.8200,21,1,1,1,SL,4786.3900,2026-04-17 03:05:00+00:00,-1.0000,46
1,BUY,2026-04-17 02:15:00+00:00,4796.3800,4790.5300,4808.0800,5.8500,4793.7300,21,1,13,1,SL,4790.5300,2026-04-17 03:00:00+00:00,-1.0000,9
2,BUY,2026-04-17 03:25:00+00:00,4792.8200,4791.7100,4795.0400,1.1100,4792.5100,21,1,1,8,TP,4795.0355,2026-04-17 03:30:00+00:00,2.0000,1
3,BUY,2026-04-17 18:20:00+00:00,4870.1700,4781.9800,5046.5500,88.1900,4867.7200,21,1,45,2,SL,4781.9800,2026-04-20 01:05:00+00:00,-1.0000,68
4,BUY,2026-04-17 19:05:00+00:00,4871.4200,4862.2700,4889.7200,9.1500,4867.6500,21,1,1,1,SL,4862.2700,2026-04-17 20:05:00+00:00,-1.0000,12


## Step 6 — Multi-EMA Comparison

آزمایش با EMA 9، 21 و 50 برای پیدا کردن بهترین پارامتر:

In [7]:
ema_results = {}

for ema_n in [EMA_FAST, EMA_MID, EMA_SLOW]:
    result_df = run_ema_pullback_backtest(df_m5, df_h1, entry_ema=ema_n)
    if result_df.empty:
        ema_results[ema_n] = {'n': 0, 'wr': 0, 'total_r': 0, 'pf': 0}
        continue
    closed = result_df[result_df['result'].isin(['TP','SL'])]
    if closed.empty:
        ema_results[ema_n] = {'n': 0, 'wr': 0, 'total_r': 0, 'pf': 0}
        continue
    n    = len(closed)
    wins = (closed['result'] == 'TP').sum()
    wr   = wins / n
    pos  = closed[closed['pnl_r'] > 0]['pnl_r'].sum()
    neg  = abs(closed[closed['pnl_r'] < 0]['pnl_r'].sum())
    pf   = pos / neg if neg > 0 else float('inf')
    ema_results[ema_n] = {
        'n': n, 'wr': wr,
        'total_r': closed['pnl_r'].sum(),
        'pf': pf,
        'df': result_df
    }

print('EMA Comparison Results:')
print(f'  {"EMA":5s} {"Trades":8s} {"WR":8s} {"Total R":10s} {"PF":8s}')
print('  ' + '-' * 45)
for ema_n, r in ema_results.items():
    wr_str = f'{r["wr"]*100:.0f}%' if r['n'] > 0 else 'N/A'
    pf_str = f'{r["pf"]:.2f}'      if r['n'] > 0 else 'N/A'
    tr_str = f'{r["total_r"]:+.1f}' if r['n'] > 0 else 'N/A'
    print(f'  EMA{ema_n:<4d} {r["n"]:8d} {wr_str:8s} {tr_str:10s} {pf_str:8s}')

KeyError: 'ema9_slope'

## Step 7 — Performance Analytics

In [ ]:
def calc_metrics(trades_df: pd.DataFrame) -> dict:
    if trades_df.empty:
        return {}
    closed = trades_df[trades_df['result'].isin(['TP', 'SL'])].copy()
    if closed.empty:
        return {}
    n   = len(closed)
    wins = (closed['result'] == 'TP').sum()
    wr  = wins / n
    closed['cum_r'] = closed['pnl_r'].cumsum()
    dd  = closed['cum_r'] - closed['cum_r'].cummax()
    pos = closed[closed['pnl_r'] > 0]['pnl_r'].sum()
    neg = abs(closed[closed['pnl_r'] < 0]['pnl_r'].sum())
    pf  = pos / neg if neg > 0 else float('inf')
    arr = (closed['result'] == 'SL').astype(int).values
    max_cl = streak = 0
    for v in arr:
        streak = (streak + 1) if v else 0
        max_cl = max(max_cl, streak)
    return {
        'total_trades'   : n,
        'wins'           : int(wins),
        'losses'         : n - int(wins),
        'win_rate'       : wr,
        'total_r'        : round(closed['pnl_r'].sum(), 3),
        'avg_r'          : round(closed['pnl_r'].mean(), 3),
        'profit_factor'  : round(pf, 3),
        'max_dd_r'       : round(dd.min(), 3),
        'max_consec_loss': max_cl,
        'expectancy'     : round(wr * RISK_REWARD - (1 - wr), 3),
        'avg_bars'       : round(closed['bars_held'].mean(), 1),
        'cum_r'          : closed['cum_r'].reset_index(drop=True),
        'drawdown'       : dd.reset_index(drop=True),
        'closed'         : closed,
    }


metrics = calc_metrics(trades_df)

if metrics and metrics.get('total_trades', 0) > 0:
    sep = '=' * 55
    print(sep)
    print(f'  PERFORMANCE — EMA{ENTRY_EMA} Pullback Reaction')
    print(sep)
    print(f'  Trades        : {metrics["total_trades"]}')
    print(f'  Wins / Losses : {metrics["wins"]} / {metrics["losses"]}')
    print(f'  Win Rate      : {metrics["win_rate"]*100:.1f}%')
    print(f'  Total R       : {metrics["total_r"]:+.2f} R')
    print(f'  Profit Factor : {metrics["profit_factor"]:.2f}')
    print(f'  Expectancy    : {metrics["expectancy"]:+.3f} R')
    print(f'  Max Drawdown  : {metrics["max_dd_r"]:.2f} R')
    print(f'  Avg Duration  : {metrics["avg_bars"]:.0f} M5 bars')
    if 'h1_trend' in trades_df.columns:
        print('\n  By H1 Trend:')
        closed = metrics['closed']
        for t in [1, -1]:
            sub = closed[closed['h1_trend'] == t]
            if sub.empty: continue
            label = 'UP  ' if t == 1 else 'DOWN'
            wr_t  = (sub['result'] == 'TP').mean()
            print(f'    Trend {label}: n={len(sub)}  WR={wr_t*100:.0f}%  AvgR={sub["pnl_r"].mean():+.3f}')
    print(sep)

## Step 8 — Visualizations

### 8.1 — EMA Chart (H1)

In [ ]:
def plot_ema_h1(df_h1: pd.DataFrame, tail_bars: int = 200) -> None:
    data = df_h1.tail(tail_bars).copy()
    fig  = go.Figure()
    fig.add_trace(go.Candlestick(
        x=data['time'], open=data['open'], high=data['high'],
        low=data['low'], close=data['close'],
        name='H1',
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350',
    ))
    colors = {
        f'ema{EMA_FAST}' : '#FFD700',
        f'ema{EMA_MID}'  : '#00BCD4',
        f'ema{EMA_SLOW}' : '#FF9800',
    }
    for col, color in colors.items():
        fig.add_trace(go.Scatter(
            x=data['time'], y=data[col],
            mode='lines', name=col.upper(),
            line=dict(color=color, width=2),
        ))
    fig.update_layout(
        title=f'XAUUSD H1 — EMA Trend Dashboard',
        xaxis_rangeslider_visible=False,
        template='plotly_dark', height=550,
    )
    fig.show()


plot_ema_h1(df_h1)

### 8.2 — Equity Curve

In [ ]:
def plot_equity(metrics: dict, ema_n: int) -> None:
    if not metrics or 'cum_r' not in metrics:
        return
    cum_r = metrics['cum_r']
    dd    = metrics['drawdown']
    closed= metrics['closed'].reset_index(drop=True)

    fig = make_subplots(rows=3, cols=1,
                        row_heights=[0.5, 0.25, 0.25],
                        subplot_titles=['Equity (R)', 'Drawdown', 'Per-Trade PnL'],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.values, mode='lines',
        line=dict(color='#00E5FF', width=2.5),
        fill='tozeroy', fillcolor='rgba(0,229,255,0.08)', name='Equity',
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.cummax().values, mode='lines',
        line=dict(color='gold', width=1, dash='dot'), name='Peak',
    ), row=1, col=1)
    fig.add_hline(y=0, line_color='gray', line_dash='dash', row=1, col=1)
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values, mode='lines',
        fill='tozeroy', fillcolor='rgba(255,23,68,0.15)',
        line=dict(color='#FF1744', width=1.5), name='DD',
    ), row=2, col=1)
    colors = ['#26a69a' if r == 'TP' else '#ef5350' for r in closed['result']]
    fig.add_trace(go.Bar(
        x=closed.index, y=closed['pnl_r'],
        marker_color=colors, name='PnL',
    ), row=3, col=1)
    fig.add_hline(y=0, line_color='gray', line_dash='dash', row=3, col=1)
    fig.update_layout(
        title=dict(
            text=(f'EMA{ema_n} Pullback — Equity<br>'
                  f'<sup>WR={metrics["win_rate"]*100:.0f}%  '
                  f'Total={metrics["total_r"]:+.1f}R  '
                  f'PF={metrics["profit_factor"]:.2f}  '
                  f'MaxDD={metrics["max_dd_r"]:.1f}R</sup>'),
            x=0.5,
        ),
        height=700, template='plotly_dark',
    )
    fig.show()


plot_equity(metrics, ENTRY_EMA)

### 8.3 — All Trades on M5 with EMAs

In [ ]:
def plot_ema_trades(df_m5: pd.DataFrame, trades_df: pd.DataFrame,
                    tail_bars: int = 1000) -> None:
    data = df_m5.tail(tail_bars).copy()
    fig  = go.Figure()
    fig.add_trace(go.Candlestick(
        x=data['time'], open=data['open'], high=data['high'],
        low=data['low'], close=data['close'],
        name='M5',
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350',
    ))

    # EMAs
    fig.add_trace(go.Scatter(
        x=data['time'], y=data[f'ema{EMA_FAST}'],
        mode='lines', name=f'EMA{EMA_FAST}',
        line=dict(color='#FFD700', width=1),
    ))
    fig.add_trace(go.Scatter(
        x=data['time'], y=data[f'ema{EMA_MID}'],
        mode='lines', name=f'EMA{EMA_MID}',
        line=dict(color='#00BCD4', width=2),
    ))
    fig.add_trace(go.Scatter(
        x=data['time'], y=data[f'ema{EMA_SLOW}'],
        mode='lines', name=f'EMA{EMA_SLOW}',
        line=dict(color='#FF9800', width=1.5),
    ))

    # Trades
    if not trades_df.empty:
        for _, t in trades_df.iterrows():
            ec   = '#26a69a' if t['direction'] == 'BUY' else '#ef5350'
            rc   = '#00E676' if t['result'] == 'TP' else '#FF1744'
            esym = 'triangle-up' if t['direction'] == 'BUY' else 'triangle-down'
            fig.add_trace(go.Scatter(
                x=[t['entry_time']], y=[t['entry_price']],
                mode='markers', showlegend=False,
                marker=dict(symbol=esym, size=12, color=rc,
                            line=dict(color='white', width=1)),
            ))

    wr = metrics.get('win_rate', 0) * 100 if metrics else 0
    tr = metrics.get('total_r', 0) if metrics else 0
    fig.update_layout(
        title=(f'EMA Pullback Reaction — XAUUSD M5  |  '
               f'WR={wr:.0f}%  Total={tr:+.1f}R'),
        xaxis_rangeslider_visible=False,
        template='plotly_dark', height=600,
        legend=dict(orientation='h', yanchor='bottom', y=1.01),
    )
    fig.show()


plot_ema_trades(df_m5, trades_df)

### 8.4 — EMA Comparison Chart

In [ ]:
def plot_ema_comparison(ema_results: dict) -> None:
    ema_ns  = sorted(ema_results.keys())
    ns      = [ema_results[e]['n']       for e in ema_ns]
    wrs     = [ema_results[e]['wr']*100  for e in ema_ns]
    pfs     = [ema_results[e]['pf']      for e in ema_ns if ema_results[e]['n'] > 0]
    trs     = [ema_results[e]['total_r'] for e in ema_ns]
    labels  = [f'EMA{e}' for e in ema_ns]

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=['Win Rate %', 'Total R', 'Profit Factor'])

    be_wr = 100 / (1 + RISK_REWARD)
    fig.add_trace(go.Bar(
        x=labels, y=wrs,
        text=[f'{w:.0f}%<br>(n={n})' for w, n in zip(wrs, ns)],
        textposition='auto', marker_color='#00BCD4',
    ), row=1, col=1)
    fig.add_hline(y=be_wr, line_dash='dash', line_color='yellow',
                  annotation_text=f'BE={be_wr:.0f}%', row=1, col=1)

    fig.add_trace(go.Bar(
        x=labels, y=trs,
        text=[f'{t:+.1f}R' for t in trs],
        textposition='auto',
        marker_color=['#26a69a' if t >= 0 else '#ef5350' for t in trs],
    ), row=1, col=2)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

    pf_valid = [ema_results[e]['pf'] if ema_results[e]['n'] > 0 else 0 for e in ema_ns]
    fig.add_trace(go.Bar(
        x=labels, y=pf_valid,
        text=[f'{p:.2f}' for p in pf_valid],
        textposition='auto', marker_color='#7C4DFF',
    ), row=1, col=3)
    fig.add_hline(y=1, line_dash='dash', line_color='yellow', row=1, col=3)

    fig.update_layout(
        title='EMA Pullback — Multi-EMA Comparison',
        template='plotly_dark', height=400, showlegend=False,
    )
    fig.show()


plot_ema_comparison(ema_results)

## Final Analysis

In [ ]:
if metrics and metrics.get('total_trades', 0) > 0:
    be_wr = 1 / (1 + RISK_REWARD)
    best_ema = max(ema_results.keys(), key=lambda e: ema_results[e].get('total_r', -999))

    print('=' * 60)
    print('  EMA PULLBACK REACTION — FINAL ANALYSIS')
    print('=' * 60)
    print(f'\n[EDGE] (Active EMA = EMA{ENTRY_EMA})')
    print(f'  Break-even WR : {be_wr*100:.1f}%')
    print(f'  Actual WR     : {metrics["win_rate"]*100:.1f}%')
    assessment = '✅ POSITIVE' if metrics['win_rate'] > be_wr else '❌ NEGATIVE'
    print(f'  Edge          : {assessment}')
    print(f'  Best EMA      : EMA{best_ema} (highest Total R)')

    print(f'\n[STRENGTHS]')
    print('  ✓ Systematic, fully rule-based')
    print('  ✓ H1 trend alignment = high quality entries')
    print('  ✓ EMA slope filter reduces false signals')
    print('  ✓ Clear SL (below/above EMA + buffer)')
    print('  ✓ High trade frequency (suitable for active trading)')

    print(f'\n[WEAKNESSES]')
    print('  ✗ Choppy markets = many false pullbacks to EMA')
    print('  ✗ EMA lags price — entries can be late in strong trends')
    print('  ✗ In ranging markets, EMA21 offers no reliable directional bias')

    print(f'\n[OPTIMIZATIONS]')
    print('  → Add ADX filter (only trade when ADX > 25 = trending)')
    print('  → Use EMA bounce on H1 for higher probability setup')
    print('  → Add price action confirmation (engulfing candle)')
    print('  → Consider EMA21 as base + require close beyond EMA9')
    print('  → Filter by time (avoid Asia session for Gold)')
    print('=' * 60)